# P03 · Sampling-based planning and trajectory time scaling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/09_planning_learning/P03_sampling_based_planning_and_trajectory_time_scal.ipynb)

**Track:** Planning and learning · **Skill axis (sheet):** Planning and learning · **Time:** about 60 min

**Skill Hub concepts:** A*, RRT, probabilistic roadmap; Trajectory time scaling

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("P03")

## Map: when grids get too big, sample

| Planner | Builds | Query type | Optimal | Good for |
|---|---|---|---|---|
| A\* on a grid | graph of all cells | single | yes (on the grid) | 2D/3D maps |
| RRT | tree grown toward random samples | single | no (RRT\* is asymptotically) | high-dimensional arms, one query |
| PRM | roadmap of random collision-free samples, connected to neighbors | many | asymptotically (PRM\*) | many queries in a static world |

**RRT loop:** sample $q_{rand}$ (with probability `goal_bias` use the goal), find the nearest tree node, step at most
`step` toward the sample, keep the new node only if the segment is collision-free, stop when within `step` of the goal.

**Time scaling** turns a geometric path $p(s)$, $s \in [0, 1]$, into a trajectory $p(s(t))$:
$$\text{cubic: } s = 3\tau^2 - 2\tau^3, \qquad \text{quintic: } s = 10\tau^3 - 15\tau^4 + 6\tau^5, \qquad \tau = t / T$$
Cubic: zero velocity at both ends, peak $\dot s = \frac{3}{2T}$, peak $|\ddot s| = \frac{6}{T^2}$, but acceleration
jumps at the ends. Quintic: zero velocity **and** acceleration at both ends, peak $\dot s = \frac{15}{8T}$.
Shortest cubic duration for distance $D$ under limits: $T = \max\left(\frac{3D}{2 v_{max}}, \sqrt{\frac{6D}{a_{max}}}\right)$.

In [ ]:
import numpy as np
CIRCLES = [(0.5, 0.5, 0.2), (0.25, 0.8, 0.12), (0.75, 0.2, 0.15)]   # (x, y, radius) in a unit square

## Task 1: segment collision check and RRT

In [ ]:
def segment_free(p, q, circles):
    """True if the segment p-q stays outside every circle."""
    raise NotImplementedError  # TODO

def rrt(start, goal, circles, step=0.05, goal_bias=0.1, max_iter=5000, rng=None):
    """Return a list of points from start to goal (goal appended last), or None."""
    raise NotImplementedError  # TODO

lab.check("rrt", segment_free, rrt)

## Task 2: time scaling

In [ ]:
def cubic(T, t):
    """Return (s, s_dot)."""
    raise NotImplementedError  # TODO

def quintic(T, t):
    """Return (s, s_dot, s_ddot)."""
    raise NotImplementedError  # TODO

def min_time_cubic(distance, vmax, amax):
    raise NotImplementedError  # TODO

lab.check("scaling", cubic, quintic, min_time_cubic)

In [ ]:
try:
    import matplotlib.pyplot as plt
    path = rrt((0.05, 0.05), (0.95, 0.95), CIRCLES, rng=np.random.default_rng(0))
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    for cx, cy, r in CIRCLES:
        ax[0].add_patch(plt.Circle((cx, cy), r, color="gray"))
    ax[0].plot(*np.array(path).T, "r.-"); ax[0].set_aspect("equal"); ax[0].set_title("RRT path")
    t = np.linspace(0, 2, 200)
    ax[1].plot(t, cubic(2, t)[1], label="cubic s_dot"); ax[1].plot(t, quintic(2, t)[1], label="quintic s_dot"); ax[1].legend()
    plt.show()
except NotImplementedError:
    print("finish Tasks 1 and 2 first")

## Checkpoint: Math
**M1 (Apply).** Cubic time scaling with $T = 2$ s: peak $\dot s$ in 1/s?

**M2 (Apply).** Quintic with $T = 2$ s: peak $\dot s$?

**M3 (Apply).** Shortest cubic duration for 1 rad with $v_{max} = 1$ rad/s and $a_{max} = 6$ rad/s$^2$?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** A pick-and-place cell plans thousands of motions per day in a fixed workcell. Which planner fits best?
- a) RRT from scratch for every query
- b) A PRM built once, then queried many times
- c) A\* on a 6D joint grid at 1 degree resolution

**K2 (Understand).** Why separate the geometric path from its timing, and why might you pick a quintic over a cubic on
a real arm?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `p03.h`).
3. In `roboprep/checks/p03.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()